In [ ]:
!nvidia-smi || true

!pip install -q tensorflow numpy scikit-learn tqdm

Fri May  8 03:26:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             33W /   70W |   12675MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from pathlib import Path

# Training size
SAMPLES_PER_CLASS = 50000
MAX_LEN = 128
BATCH_SIZE = 256
EPOCHS = 40
SEED = 42

# Output paths
DATA_DIR = Path("/content/data")
NDJSON_DIR = DATA_DIR / "quickdraw_simplified"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = Path("/content/quickdraw_stroke_tflite")

NDJSON_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Start with clean, visually distinct categories.
CLASSES = [
    "The Mona Lisa",           # Verified
    "The Great Wall of China", # Verified
    "The Eiffel Tower",        # Verified
    "piano",                   # FIXED: Official name (not 'grand piano')
    "calculator",              # SWAP: Replaces 'spreadsheet' (hard to draw)
    "binoculars",              # SWAP: Replaces 'microscope'
    "skull",                   # SWAP: Replaces 'skeleton'
    "sleeping bag",            # Verified
    "roller coaster",          # Verified
    "chandelier",              # Verified
    "brain",                   # Verified
    "animal migration",        # Verified
    "stethoscope",             # Verified
    "power outlet",            # Verified
    "scorpion",                # Verified
    "bulldozer",               # Verified
    "saxophone",               # Verified
    "violin",                  # Verified
    "fire hydrant",            # Verified
    "garden hose",             # Verified
    "windmill",                # Verified
    "cruise ship",             # Verified
    "hourglass",               # Verified
    "backpack",                # Verified
    "dragon",                  # Verified
    "helicopter",              # Verified
    "harp",                    # Verified
    "hedgehog",                # Verified
    "camouflage",              # Verified (Very hard for humans!)
    "yoga"                     # Verified (Abstract/Hard)
]

print("Classes:", len(CLASSES))

Classes: 30


In [ ]:
import urllib.request
from pathlib import Path

# Make sure the directory exists
NDJSON_DIR.mkdir(parents=True, exist_ok=True)

for class_name in CLASSES:
    # 1. We use the https link instead of gs:// because it's easier to fix spaces
    safe_name = class_name.replace(' ', '%20')
    src_url = f"https://storage.googleapis.com/quickdraw_dataset/full/simplified/{safe_name}.ndjson"

    # 2. This is where the file will be saved
    dst = NDJSON_DIR / f"{class_name}.ndjson"

    if dst.exists():
        print(f"Already exists: {dst}")
        continue

    print(f"Downloading: {class_name}")
    try:
        # This replaces the 'subprocess.run' line and is much more stable
        urllib.request.urlretrieve(src_url, dst)
    except Exception as e:
        print(f"Error downloading {class_name}: {e}")

print("Download complete.")

Already exists: /content/data/quickdraw_simplified/The Mona Lisa.ndjson
Already exists: /content/data/quickdraw_simplified/The Great Wall of China.ndjson
Already exists: /content/data/quickdraw_simplified/The Eiffel Tower.ndjson
Already exists: /content/data/quickdraw_simplified/piano.ndjson
Already exists: /content/data/quickdraw_simplified/calculator.ndjson
Already exists: /content/data/quickdraw_simplified/binoculars.ndjson
Already exists: /content/data/quickdraw_simplified/skull.ndjson
Already exists: /content/data/quickdraw_simplified/sleeping bag.ndjson
Already exists: /content/data/quickdraw_simplified/roller coaster.ndjson
Already exists: /content/data/quickdraw_simplified/chandelier.ndjson
Already exists: /content/data/quickdraw_simplified/brain.ndjson
Already exists: /content/data/quickdraw_simplified/animal migration.ndjson
Already exists: /content/data/quickdraw_simplified/stethoscope.ndjson
Already exists: /content/data/quickdraw_simplified/power outlet.ndjson
Already exis

In [ ]:
import json
import random
import numpy as np
import gc  # <--- Added for garbage collection (memory management)
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# Ensure SEED and other constants are defined earlier in your notebook
random.seed(SEED)
np.random.seed(SEED)

def normalize_strokes(drawing):
    """
    QuickDraw simplified drawing format normalization.
    """
    points = []

    for stroke in drawing:
        if len(stroke) != 2:
            continue
        xs, ys = stroke
        for x, y in zip(xs, ys):
            points.append((float(x), float(y)))

    if not points:
        return []

    xs_all = [p[0] for p in points]
    ys_all = [p[1] for p in points]

    min_x, max_x = min(xs_all), max(xs_all)
    min_y, max_y = min(ys_all), max(ys_all)

    width = max(max_x - min_x, 1.0)
    height = max(max_y - min_y, 1.0)

    scale = 255.0 / max(width, height)

    normalized = []

    for stroke in drawing:
        if len(stroke) != 2:
            continue
        xs, ys = stroke
        new_stroke = []
        for x, y in zip(xs, ys):
            nx = (float(x) - min_x) * scale
            ny = (float(y) - min_y) * scale
            new_stroke.append((nx, ny))

        if len(new_stroke) >= 2:
            normalized.append(new_stroke)

    return normalized

def strokes_to_sequence(drawing, max_len=MAX_LEN):
    """
    Converts QuickDraw strokes into a fixed tensor: [max_len, 5]
    [dx, dy, pen_down, pen_up, end]
    """
    strokes = normalize_strokes(drawing)
    seq = []
    prev_x = None
    prev_y = None

    for stroke in strokes:
        for i, (x, y) in enumerate(stroke):
            if prev_x is None:
                dx = 0.0
                dy = 0.0
            else:
                dx = x - prev_x
                dy = y - prev_y

            is_last_point = i == len(stroke) - 1

            pen_down = 0.0 if is_last_point else 1.0
            pen_up = 1.0 if is_last_point else 0.0
            end = 0.0

            seq.append([
                dx / 255.0,
                dy / 255.0,
                pen_down,
                pen_up,
                end,
            ])

            prev_x = x
            prev_y = y

            if len(seq) >= max_len - 1:
                break
        if len(seq) >= max_len - 1:
            break

    # End token
    seq.append([0.0, 0.0, 0.0, 0.0, 1.0])

    # Pad
    while len(seq) < max_len:
        seq.append([0.0, 0.0, 0.0, 0.0, 0.0])

    return np.array(seq[:max_len], dtype=np.float32)

# --- THE FIXED FUNCTION ---
def read_class_samples(class_name, label_id, limit):
    path = NDJSON_DIR / f"{class_name}.ndjson"
    xs = []
    ys = []

    if not path.exists():
        print(f"⚠️ Warning: File not found for {class_name}")
        return xs, ys

    with open(path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"Processing {class_name}"):
            line = line.strip()
            if not line: # Skip empty lines
                continue

            try:
                # THE FIX: Wrap json.loads in a try-except block
                row = json.loads(line)
            except json.JSONDecodeError:
                # If a line is corrupt, skip it and continue to the next
                continue

            if not row.get("recognized", False):
                continue

            drawing = row.get("drawing")
            if not drawing:
                continue

            x = strokes_to_sequence(drawing)
            xs.append(x)
            ys.append(label_id)

            if len(xs) >= limit:
                break

    return xs, ys
# ---------------------------

all_x = []
all_y = []

label_to_id = {name: i for i, name in enumerate(CLASSES)}

for class_name in CLASSES:
    label_id = label_to_id[class_name]
    xs, ys = read_class_samples(class_name, label_id, SAMPLES_PER_CLASS)

    all_x.extend(xs)
    all_y.extend(ys)

print(f"\nTotal samples loaded: {len(all_x)}. Safely converting to NumPy arrays...")

# --- THE MEMORY HACK ---
# 1. Create the final empty array first
x = np.zeros((len(all_x), MAX_LEN, 5), dtype=np.float32)

# 2. Move data over one by one and delete the original to save RAM
for i in tqdm(range(len(all_x)), desc="Stacking Arrays"):
    x[i] = all_x[i]
    all_x[i] = None  # Free up memory immediately

# 3. Destroy the old list completely
del all_x
gc.collect()

# Do the same for labels
y = np.array(all_y, dtype=np.int64)
del all_y
gc.collect()
# -----------------------

print("\n--- Processing Complete ---")
print("X shape:", x.shape)
print("Y shape:", y.shape)
print("Total Classes:", len(CLASSES))

Processing The Mona Lisa: 54468it [00:18, 2998.75it/s]
Processing The Great Wall of China: 56328it [00:06, 8423.26it/s]
Processing The Eiffel Tower: 51678it [00:05, 8775.63it/s]
Processing piano: 51279it [00:06, 7554.44it/s]
Processing calculator: 53338it [00:07, 6818.29it/s]
Processing binoculars: 57123it [00:06, 8354.85it/s]
Processing skull: 52941it [00:08, 6565.93it/s]
Processing sleeping bag: 59311it [00:05, 10028.97it/s]
Processing roller coaster: 58931it [00:07, 8132.59it/s]
Processing chandelier: 57479it [00:06, 8426.56it/s]
Processing brain: 56695it [00:08, 6312.46it/s]
Processing animal migration: 59360it [00:07, 7952.03it/s]
Processing stethoscope: 58309it [00:07, 7942.42it/s]
Processing power outlet: 57494it [00:07, 7880.13it/s]
Processing scorpion: 57657it [00:08, 6716.79it/s]
Processing bulldozer: 56598it [00:07, 8080.08it/s]
Processing saxophone: 53935it [00:07, 7497.59it/s]
Processing violin: 55630it [00:06, 8905.60it/s]
Processing fire hydrant: 59332it [00:07, 7976.83i


Total samples loaded: 1500000. Safely converting to NumPy arrays...


Stacking Arrays: 100%|██████████| 1500000/1500000 [00:05<00:00, 284490.45it/s]



--- Processing Complete ---
X shape: (1500000, 128, 5)
Y shape: (1500000,)
Total Classes: 30


In [ ]:
import gc # Garbage Collector
import json
import numpy as np

print("Bypassing Scikit-Learn to save RAM...")

samples_per_class = SAMPLES_PER_CLASS
num_classes = len(CLASSES)

train_idx, val_idx, test_idx = [], [], []

# Manually calculate the 80% Train / 10% Val / 10% Test split indices
for i in range(num_classes):
    start = i * samples_per_class
    end = start + samples_per_class

    idx = np.random.permutation(np.arange(start, end))

    train_end = int(0.8 * samples_per_class)
    val_end = int(0.9 * samples_per_class)

    train_idx.extend(idx[:train_end])
    val_idx.extend(idx[train_end:val_end])
    test_idx.extend(idx[val_end:])

# Shuffle the combined indices
np.random.seed(SEED)
np.random.shuffle(train_idx)
np.random.shuffle(val_idx)
np.random.shuffle(test_idx)

print("Using Memory-Mapped Files (Direct to Disk) to completely bypass RAM limits...")

# --- 1. TEST SET ---
print("Writing Test Set directly to disk...")
x_test = np.lib.format.open_memmap(PROCESSED_DIR / "x_test.npy", mode='w+', dtype=np.float32, shape=(len(test_idx), MAX_LEN, 5))
y_test = np.lib.format.open_memmap(PROCESSED_DIR / "y_test.npy", mode='w+', dtype=np.int64, shape=(len(test_idx),))

chunk_size = 50000
for i in range(0, len(test_idx), chunk_size):
    chunk = test_idx[i:i+chunk_size]
    x_test[i:i+chunk_size] = x[chunk]
    y_test[i:i+chunk_size] = y[chunk]
del x_test, y_test # Flush to disk safely
gc.collect()

# --- 2. VAL SET ---
print("Writing Validation Set directly to disk...")
x_val = np.lib.format.open_memmap(PROCESSED_DIR / "x_val.npy", mode='w+', dtype=np.float32, shape=(len(val_idx), MAX_LEN, 5))
y_val = np.lib.format.open_memmap(PROCESSED_DIR / "y_val.npy", mode='w+', dtype=np.int64, shape=(len(val_idx),))

for i in range(0, len(val_idx), chunk_size):
    chunk = val_idx[i:i+chunk_size]
    x_val[i:i+chunk_size] = x[chunk]
    y_val[i:i+chunk_size] = y[chunk]
del x_val, y_val # Flush to disk safely
gc.collect()

# --- 3. TRAIN SET ---
print("Writing Train Set directly to disk (This is the big one)...")
x_train = np.lib.format.open_memmap(PROCESSED_DIR / "x_train.npy", mode='w+', dtype=np.float32, shape=(len(train_idx), MAX_LEN, 5))
y_train = np.lib.format.open_memmap(PROCESSED_DIR / "y_train.npy", mode='w+', dtype=np.int64, shape=(len(train_idx),))

for i in range(0, len(train_idx), chunk_size):
    chunk = train_idx[i:i+chunk_size]
    x_train[i:i+chunk_size] = x[chunk]
    y_train[i:i+chunk_size] = y[chunk]
del x_train, y_train # Flush to disk safely

# 🚨 CRITICAL: Destroy the giant original array
print("Destroying the original array to free RAM...")
del x, y
gc.collect()

with open(PROCESSED_DIR / "labels.json", "w", encoding="utf-8") as f:
    json.dump(CLASSES, f, indent=2)

print("✅ Splitting Complete! Re-loading references safely...")

# Load them back in 'r' (read-only memory-map mode) for the training cell!
x_train = np.load(PROCESSED_DIR / "x_train.npy", mmap_mode='r')
y_train = np.load(PROCESSED_DIR / "y_train.npy", mmap_mode='r')
x_val = np.load(PROCESSED_DIR / "x_val.npy", mmap_mode='r')
y_val = np.load(PROCESSED_DIR / "y_val.npy", mmap_mode='r')

print("Train:", x_train.shape, y_train.shape)
print("Val:", x_val.shape, y_val.shape)
print("Ready for Training!")

Bypassing Scikit-Learn to save RAM...
Using Memory-Mapped Files (Direct to Disk) to completely bypass RAM limits...
Writing Test Set directly to disk...
Writing Validation Set directly to disk...
Writing Train Set directly to disk (This is the big one)...
Destroying the original array to free RAM...
✅ Splitting Complete! Re-loading references safely...
Train: (1200000, 128, 5) (1200000,)
Val: (150000, 128, 5) (150000,)
Ready for Training!


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

num_classes = len(CLASSES)

inputs = keras.Input(shape=(MAX_LEN, 5), name="stroke_input")

x_in = layers.Conv1D(64, kernel_size=5, padding="same", activation="relu")(inputs)
x_in = layers.BatchNormalization()(x_in)
x_in = layers.MaxPooling1D(pool_size=2)(x_in)

x_in = layers.Conv1D(128, kernel_size=5, padding="same", activation="relu")(x_in)
x_in = layers.BatchNormalization()(x_in)
x_in = layers.MaxPooling1D(pool_size=2)(x_in)

x_in = layers.Conv1D(256, kernel_size=3, padding="same", activation="relu")(x_in)
x_in = layers.BatchNormalization()(x_in)

x_in = layers.GlobalAveragePooling1D()(x_in)

x_in = layers.Dense(256, activation="relu")(x_in)
# 👇 THIS IS THE TWEAK: Changed from 0.30 to 0.50 to handle 1.5M drawings better
x_in = layers.Dropout(0.50)(x_in)

outputs = layers.Dense(num_classes, activation="softmax", name="class_probs")(x_in)

model = keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3"),
    ],
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ stroke_input (InputLayer)       │ (None, 128, 5)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 128, 64)        │         1,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 64, 128)        │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 64, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 32, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 32, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ class_probs (Dense)             │ (None, 30)             │         7,710 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 216,606 (846.12 KB)

 Trainable params: 215,710 (842.62 KB)

 Non-trainable params: 896 (3.50 KB)

In [ ]:
import gc
import numpy as np

# 1. Free up RAM by deleting the test set (we don't need it until training is over)
if 'x_test' in locals():
    del x_test
    del y_test
    gc.collect()

print("Setting up Memory-Safe Data Generators...")

# 2. Create the Generator Pipeline
class MemorySafeGenerator(keras.utils.Sequence):
    def __init__(self, x_data, y_data, batch_size):
        self.x = x_data
        self.y = y_data
        self.batch_size = batch_size
        self.indices = np.arange(len(self.x))
        np.random.shuffle(self.indices) # Shuffle data before starting

    def __len__(self):
        return int(np.ceil(len(self.x) / float(self.batch_size)))

    def __getitem__(self, idx):
        # Fetch just the specific 256 drawings for this batch
        batch_indices = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        return self.x[batch_indices], self.y[batch_indices]

    def on_epoch_end(self):
        # Re-shuffle at the end of every epoch
        np.random.shuffle(self.indices)

# 3. Initialize the generators
train_gen = MemorySafeGenerator(x_train, y_train, BATCH_SIZE)
val_gen = MemorySafeGenerator(x_val, y_val, BATCH_SIZE)

# 4. Standard Callbacks
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_DIR / "best.keras"),
        monitor="val_top3",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_top3",
        mode="max",
        patience=6,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_top3",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1,
    ),
]

print("Starting training safely...")

# 5. Train using the generators!
history = model.fit(
    train_gen,                # <--- Feeding via the generator
    validation_data=val_gen,  # <--- Feeding via the generator
    epochs=EPOCHS,
    callbacks=callbacks,
)

model.save(MODEL_DIR / "final.keras")

with open(MODEL_DIR / "labels.json", "w", encoding="utf-8") as f:
    json.dump(CLASSES, f, indent=2)

print("✅ Saved Keras model and labels. Training Complete!")

Setting up Memory-Safe Data Generators...
Starting training safely...
Epoch 1/40
4686/4688 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6969 - loss: 1.0373 - top3: 0.8569
Epoch 1: val_top3 improved from None to 0.96809, saving model to /content/quickdraw_stroke_tflite/best.keras

Epoch 1: finished saving model to /content/quickdraw_stroke_tflite/best.keras
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 70s 13ms/step - accuracy: 0.8051 - loss: 0.6592 - top3: 0.9304 - val_accuracy: 0.8758 - val_loss: 0.4074 - val_top3: 0.9681 - learning_rate: 0.0010
Epoch 2/40
4684/4688 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8829 - loss: 0.3906 - top3: 0.9702
Epoch 2: val_top3 improved from 0.96809 to 0.97591, saving model to /content/quickdraw_stroke_tflite/best.keras

Epoch 2: finished saving model to /content/quickdraw_stroke_tflite/best.keras
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 47s 10ms/step - accuracy: 0.8871 - loss: 0.3762 - top3: 0.9717 - val_accuracy: 0.8977 - val_loss: 0.3373 - val_top3: 0.9759 - learnin

In [ ]:
import numpy as np
from tensorflow import keras

print("Loading test data back from disk...")
# Load them back in 'r' (read-only memory-map mode) to prevent a sudden RAM crash!
x_test = np.load(PROCESSED_DIR / "x_test.npy", mmap_mode='r')
y_test = np.load(PROCESSED_DIR / "y_test.npy", mmap_mode='r')

print("Loading best model...")
best_model = keras.models.load_model(MODEL_DIR / "best.keras")

print("Evaluating...")
loss, acc, top3 = best_model.evaluate(x_test, y_test, batch_size=BATCH_SIZE)

print(f"\n--- Final Results ---")
print(f"Test loss: {loss:.4f}")
print(f"Test top-1 accuracy: {acc:.4f} ({(acc*100):.1f}%)")
print(f"Test top-3 accuracy: {top3:.4f} ({(top3*100):.1f}%)")

Loading test data back from disk...
Loading best model...
Evaluating...
586/586 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9429 - loss: 0.1911 - top3: 0.9885

--- Final Results ---
Test loss: 0.1911
Test top-1 accuracy: 0.9429 (94.3%)
Test top-3 accuracy: 0.9885 (98.8%)


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
tflite_model = converter.convert()

tflite_path = MODEL_DIR / "quickdraw_stroke_model_float32.tflite"

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print("Saved:", tflite_path)
print("Size MB:", tflite_path.stat().st_size / 1024 / 1024)

Saved artifact at '/tmp/tmpw8oga4r8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 5), dtype=tf.float32, name='stroke_input')
Output Type:
  TensorSpec(shape=(None, 30), dtype=tf.float32, name=None)
Captures:
  139623337172560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623337174864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623337169104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623332659472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623337174672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623337170640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623332659280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623332663696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623332660048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623332661200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623332664272: T

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_float16 = converter.convert()

tflite_float16_path = MODEL_DIR / "quickdraw_stroke_model_float16.tflite"

with open(tflite_float16_path, "wb") as f:
    f.write(tflite_float16)

print("Saved:", tflite_float16_path)
print("Size MB:", tflite_float16_path.stat().st_size / 1024 / 1024)

Saved artifact at '/tmp/tmpw9lradl9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 5), dtype=tf.float32, name='stroke_input')
Output Type:
  TensorSpec(shape=(None, 30), dtype=tf.float32, name=None)
Captures:
  139623337172560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623337174864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623337169104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623332659472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623337174672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623337170640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623332659280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623332663696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623332660048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623332661200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139623332664272: T

In [ ]:
import numpy as np
import tensorflow as tf

TFLITE_PATH = str(MODEL_DIR / "quickdraw_stroke_model_float32.tflite")

interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:")
print(input_details)

print("Output details:")
print(output_details)

sample = x_test[0:1].astype(np.float32)

interpreter.set_tensor(input_details[0]["index"], sample)
interpreter.invoke()

output = interpreter.get_tensor(output_details[0]["index"])[0]

top_ids = output.argsort()[-3:][::-1]

print("True label:", CLASSES[y_test[0]])
print("Top 3:")
for idx in top_ids:
    print(CLASSES[idx], float(output[idx]))

Input details:
[{'name': 'serving_default_stroke_input:0', 'index': 0, 'shape': array([  1, 128,   5], dtype=int32), 'shape_signature': array([ -1, 128,   5], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Output details:
[{'name': 'StatefulPartitionedCall_1:0', 'index': 48, 'shape': array([ 1, 30], dtype=int32), 'shape_signature': array([-1, 30], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
True label: The Great Wall of China
Top 3:
The Great Wall of China 0.9913557171821594
piano 0.00791475921869278
roller coaster 0.0007234702934511006


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
model_info = {
    "model_name": "quickdraw_stroke_model",
    "input_shape": [1, MAX_LEN, 5],
    "input_dtype": "float32",
    "output_shape": [1, len(CLASSES)],
    "output_dtype": "float32",
    "classes": CLASSES,
    "class_count": len(CLASSES),
    "sequence_format": ["dx", "dy", "pen_down", "pen_up", "end"],
    "dx_dy_scaled_by": 255.0,
}

with open(MODEL_DIR / "model_info.json", "w", encoding="utf-8") as f:
    json.dump(model_info, f, indent=2)

print(json.dumps(model_info, indent=2))

{
  "model_name": "quickdraw_stroke_model",
  "input_shape": [
    1,
    128,
    5
  ],
  "input_dtype": "float32",
  "output_shape": [
    1,
    30
  ],
  "output_dtype": "float32",
  "classes": [
    "The Mona Lisa",
    "The Great Wall of China",
    "The Eiffel Tower",
    "piano",
    "calculator",
    "binoculars",
    "skull",
    "sleeping bag",
    "roller coaster",
    "chandelier",
    "brain",
    "animal migration",
    "stethoscope",
    "power outlet",
    "scorpion",
    "bulldozer",
    "saxophone",
    "violin",
    "fire hydrant",
    "garden hose",
    "windmill",
    "cruise ship",
    "hourglass",
    "backpack",
    "dragon",
    "helicopter",
    "harp",
    "hedgehog",
    "camouflage",
    "yoga"
  ],
  "class_count": 30,
  "sequence_format": [
    "dx",
    "dy",
    "pen_down",
    "pen_up",
    "end"
  ],
  "dx_dy_scaled_by": 255.0
}


In [ ]:
!cd /content && zip -r quickdraw_stroke_tflite_export.zip quickdraw_stroke_tflite

from google.colab import files
files.download("/content/quickdraw_stroke_tflite_export.zip")

updating: quickdraw_stroke_tflite/ (stored 0%)
updating: quickdraw_stroke_tflite/best.keras (deflated 10%)
updating: quickdraw_stroke_tflite/quickdraw_stroke_model_float16.tflite (deflated 10%)
updating: quickdraw_stroke_tflite/final.keras (deflated 10%)
updating: quickdraw_stroke_tflite/labels.json (deflated 48%)
updating: quickdraw_stroke_tflite/model_info.json (deflated 54%)
updating: quickdraw_stroke_tflite/quickdraw_stroke_model_float32.tflite (deflated 8%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>